## Preprocessing — cleaning and exporting dataset

In [ ]:
import kagglehub
import pandas as pd

from src.preprocessing.cleaning import clean_text, filter_short_articles

### Load raw data

In [ ]:
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

true_df = pd.read_csv(f"{path}/True.csv")
fake_df = pd.read_csv(f"{path}/Fake.csv")

true_df["label"] = 1
fake_df["label"] = 0

true_df.drop(columns=["title", "subject", "date"], inplace=True)
fake_df.drop(columns=["title", "subject", "date"], inplace=True)

news_df = pd.concat([true_df, fake_df], ignore_index=True)

### Remove duplicates

In [ ]:
news_df.drop_duplicates(inplace=True)
news_df.duplicated().sum()

### Clean text

Apply the cleaning pipeline from `src/preprocessing/cleaning.py`:
1. Remove wire service markers (Reuters, AP, AFP) and dateline prefixes
2. Remove URLs, Twitter pic links, @mentions
3. Remove photo/video credit templates (keep individual content words)
4. Remove HTML/scraping artifacts
5. Remove site-specific templates (21st Century Wire)
6. Fix malformed date-word joins
7. Collapse whitespace, remove single-character words, lowercase

In [ ]:
news_df["text"] = news_df["text"].apply(clean_text)

print(f"Shape after cleaning: {news_df.shape}")
news_df.head()

In [ ]:
before_count = len(news_df)
news_df = filter_short_articles(news_df, min_words=10)
after_count = len(news_df)

print(f"Dropped {before_count - after_count} articles with < 10 words")
print(f"Final shape: {news_df.shape}")
print(f"\nClass distribution:")
print(news_df["label"].value_counts().rename({0: "fake", 1: "real"}))

### Export cleaned dataset

In [ ]:
news_df.to_csv("../../data/cleaned_news_dataset.csv", index=False)